# LeetCode 883 — Projection Area of 3D Shapes bằng Python

**Link đề:** https://leetcode.com/problems/projection-area-of-3d-shapes/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Geometry, Matrix

```python
class Solution:
    def projectionArea(self, grid: list[list[int]]) -> int:
        ...
```

## Mục tiêu học
1. Đọc hiểu bài hình học 3D bằng cách **quy về các phép đếm trên ma trận 2D**.
2. Nắm ma trận trong Python: `grid[i][j]`, duyệt hàng/cột, `max` của hàng, **chuyển vị** bằng `zip(*grid)`.
3. Đi từ cách mô phỏng "đổ bóng từng khối lập phương" (baseline bằng `set`) tới công thức gọn: *đáy = số ô khác 0, mặt trước = tổng max mỗi hàng, mặt bên = tổng max mỗi cột*.
4. Viết lời giải một vòng lặp `O(n²)`, trace từng hàng, chứng minh và test đối chiếu.
5. Phân biệt bài này với bài anh em **892 Surface Area of 3D Shapes**, và luyện 6 bài tập.

## 1. Đề bài và ví dụ

Có lưới `n × n`. Tại ô `(i, j)` ta đặt một **tháp** gồm `v = grid[i][j]` khối lập phương `1×1×1` chồng lên nhau.

Ta nhìn **bóng chiếu** (projection) của cả khối lên 3 mặt phẳng:
- mặt **xy** — nhìn từ **trên xuống** (top),
- mặt **yz** và mặt **zx** — nhìn từ **phía trước** và **bên cạnh**.

Trả về **tổng diện tích** của 3 bóng chiếu.

| grid | Output |
|---|---:|
| `[[1,2],[3,4]]` | 17 |
| `[[2]]` | 5 |
| `[[1,0],[0,2]]` | 8 |

**Constraints:** `n == grid.length == grid[i].length`, `1 <= n <= 50`, `0 <= grid[i][j] <= 50`.

### 1.1. Hình dung ví dụ `[[1,2],[3,4]]`

```text
Nhìn từ trên (xy): mỗi ô có tháp (v > 0) để lại 1 ô bóng.
    [1][2]
    [3][4]        -> 4 ô khác 0  -> diện tích 4

Nhìn dọc theo hàng (thấy chiều cao lớn nhất của mỗi HÀNG):
    hàng 0: max(1,2) = 2
    hàng 1: max(3,4) = 4      -> 2 + 4 = 6

Nhìn dọc theo cột (thấy chiều cao lớn nhất của mỗi CỘT):
    cột 0: max(1,3) = 3
    cột 1: max(2,4) = 4       -> 3 + 4 = 7

Tổng: 4 + 6 + 7 = 17 ✔
```

Bóng chỉ quan tâm "có che hay không", không cộng dồn: tháp thấp đứng sau tháp cao **bị che hoàn toàn**.

## 2. Nền tảng Python cần biết

### 2.1. Ma trận = list các list

`grid[i]` là **hàng** `i` (một list), `grid[i][j]` là phần tử ở hàng `i`, cột `j`. `len(grid)` là số hàng.

In [1]:
grid = [[1, 2],
        [3, 4]]
print("Số hàng n =", len(grid))
print("Hàng 0:", grid[0])
print("grid[1][0] =", grid[1][0], "(hàng 1, cột 0)")
for i in range(len(grid)):
    for j in range(len(grid[i])):
        print(f"  ô ({i},{j}) có tháp cao {grid[i][j]}")

Số hàng n = 2
Hàng 0: [1, 2]
grid[1][0] = 3 (hàng 1, cột 0)
  ô (0,0) có tháp cao 1
  ô (0,1) có tháp cao 2
  ô (1,0) có tháp cao 3
  ô (1,1) có tháp cao 4


### 2.2. `max` của một hàng và của một cột

- Max của hàng `i`: `max(grid[i])` — vì hàng chính là một list.
- Max của cột `j`: phải gom các `grid[i][j]` theo `i`: `max(grid[i][j] for i in range(n))`.

Cú pháp `(... for i in range(n))` là **generator expression**: sinh lần lượt các giá trị, đưa thẳng vào `max`/`sum` mà không tạo list trung gian.

In [2]:
n = len(grid)
print("Max từng hàng:", [max(row) for row in grid])
print("Max từng cột :", [max(grid[i][j] for i in range(n)) for j in range(n)])

Max từng hàng: [2, 4]
Max từng cột : [3, 4]


### 2.3. Chuyển vị bằng `zip(*grid)`

- `*grid` "bung" list ra thành các đối số: `zip(*[[1,2],[3,4]])` giống `zip([1,2], [3,4])`.
- `zip` ghép phần tử cùng vị trí: `(1,3)`, `(2,4)` — chính là **các cột**!

Vậy `zip(*grid)` cho ta duyệt từng cột như duyệt hàng.

In [3]:
print("zip(*grid) ->", list(zip(*grid)))
print("Max từng cột (zip):", [max(col) for col in zip(*grid)])

g3 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("Chuyển vị 3x3:", [list(col) for col in zip(*g3)])

zip(*grid) -> [(1, 3), (2, 4)]
Max từng cột (zip): [3, 4]
Chuyển vị 3x3: [[1, 4, 7], [2, 5, 8], [3, 6, 9]]


### 2.4. Đếm số phần tử thỏa điều kiện: `sum(1 for ... if ...)` và `sum(bool)`

Trong Python, `True` tương đương `1`, `False` tương đương `0`. Nên `sum(v > 0 for v in row)` đếm số ô khác 0 trong hàng.

In [4]:
row = [3, 0, 5, 0]
print("True + True =", True + True)
print("Số ô > 0:", sum(v > 0 for v in row))
print("Cách khác:", sum(1 for v in row if v > 0))
print("Toàn lưới [[1,0],[0,2]]:", sum(v > 0 for r in [[1, 0], [0, 2]] for v in r))

True + True = 2
Số ô > 0: 2
Cách khác: 2
Toàn lưới [[1,0],[0,2]]: 2


Generator lồng hai `for` đọc từ **trái sang phải** giống hai vòng lồng nhau: `for r in grid` (ngoài) rồi `for v in r` (trong).

## 3. Cách 1 — Baseline: mô phỏng từng khối lập phương và "in bóng" vào `set`

Không cần nghĩ công thức: cứ đặt **từng khối lập phương** vào tọa độ `(x, y, z)` với `x = i`, `y = j`, `z = 0..v-1`. Mỗi khối để lại:
- bóng trên mặt xy: điểm `(x, y)`,
- bóng trên mặt yz: điểm `(y, z)`,
- bóng trên mặt zx: điểm `(z, x)`.

Bóng của nhiều khối có thể **trùng nhau** → dùng **`set`** (tập hợp: tự loại phần tử trùng). Diện tích mỗi mặt = số điểm khác nhau trong set.

In [5]:
s = set()
s.add((0, 1))
s.add((0, 1))       # trùng -> không thêm
s.add((2, 3))
print(s, "| số phần tử:", len(s))

{(0, 1), (2, 3)} | số phần tử: 2


In [6]:
def projection_area_simulate(grid: list[list[int]]) -> int:
    xy, yz, zx = set(), set(), set()
    n = len(grid)
    for x in range(n):
        for y in range(n):
            for z in range(grid[x][y]):        # từng khối trong tháp
                xy.add((x, y))
                yz.add((y, z))
                zx.add((z, x))
    return len(xy) + len(yz) + len(zx)

for g in ([[1, 2], [3, 4]], [[2]], [[1, 0], [0, 2]]):
    print(g, "->", projection_area_simulate(g))

[[1, 2], [3, 4]] -> 17
[[2]] -> 5
[[1, 0], [0, 2]] -> 8


### 3.1. Đánh giá baseline

Số khối tối đa: `n² · 50` = `2500 · 50 = 125 000` → vẫn chạy nhanh. Nhưng:
- Thời gian `O(n² · H)` với `H` = chiều cao tối đa, bộ nhớ `O(n² + n·H)` cho các set.
- Nếu chiều cao lên tới `10^9` thì không thể mô phỏng.

Nhìn vào cách các set được điền, ta thấy được **công thức** ở mục sau.

## 4. Insight: mỗi bóng chiếu là một phép đếm đơn giản

**Mặt xy (từ trên):** điểm `(x, y)` có bóng ⇔ tháp ở ô đó cao ≥ 1. → diện tích = **số ô `grid[i][j] > 0`**.

**Mặt yz (nhìn dọc theo trục x):** với cột `y` cố định, điểm `(y, z)` có bóng ⇔ tồn tại hàng `x` có tháp cao > `z`. Các `z` được phủ là `0 .. max_x grid[x][y] - 1` → mỗi **cột** `y` đóng góp **max của cột đó**.

**Mặt zx:** tương tự, mỗi **hàng** `x` đóng góp **max của hàng đó**.

```text
Đáp án = (#ô > 0) + Σ max(hàng) + Σ max(cột)
```

In [7]:
grid = [[1, 2], [3, 4]]
top = sum(v > 0 for row in grid for v in row)
rows = sum(max(row) for row in grid)
cols = sum(max(col) for col in zip(*grid))
print("top =", top, "| tổng max hàng =", rows, "| tổng max cột =", cols, "| tổng =", top + rows + cols)

top = 4 | tổng max hàng = 6 | tổng max cột = 7 | tổng = 17


### 4.1. Cẩn thận ô có giá trị 0

Ô `0` không có tháp → **không** để bóng trên mặt trên. Nhưng nó cũng không làm hỏng max hàng/cột (max ≥ 0). Đây là chỗ khác biệt duy nhất giữa "số ô" và "số ô > 0".

Ví dụ `[[1,0],[0,2]]`: top = 2 (không phải 4), max hàng = 1 + 2, max cột = 1 + 2 → 2 + 3 + 3 = 8 ✔.

## 5. Cách 2 — Ba phép tính tách rời (công thức trực tiếp)

In [8]:
def projection_area_formula(grid: list[list[int]]) -> int:
    top = sum(v > 0 for row in grid for v in row)
    front = sum(max(row) for row in grid)            # max mỗi hàng
    side = sum(max(col) for col in zip(*grid))       # max mỗi cột
    return top + front + side

for g in ([[1, 2], [3, 4]], [[2]], [[1, 0], [0, 2]]):
    print(g, "->", projection_area_formula(g))

[[1, 2], [3, 4]] -> 17
[[2]] -> 5
[[1, 0], [0, 2]] -> 8


Cách này duyệt lưới **3 lần** (mỗi lần `O(n²)`). Gọn, dễ đọc — rất phù hợp nộp bài.

## 6. Cách 3 — Một vòng lặp đôi, tính cả max hàng và max cột cùng lúc

Mẹo: lưới là **vuông** (`n × n`). Trong vòng `for i`, `for j`:
- `grid[i][j]` duyệt **hàng** `i`,
- `grid[j][i]` duyệt **cột** `i` (đổi vai trò chỉ số).

Nên cùng một vòng `j` ta cập nhật được cả `row_max` của hàng `i` và `col_max` của cột `i`.

In [9]:
def projection_area_one_pass(grid: list[list[int]]) -> int:
    n = len(grid)
    total = 0
    for i in range(n):
        row_max = 0
        col_max = 0
        for j in range(n):
            if grid[i][j] > 0:
                total += 1                       # bóng mặt trên
            row_max = max(row_max, grid[i][j])   # hàng i
            col_max = max(col_max, grid[j][i])   # cột i
        total += row_max + col_max
    return total

for g in ([[1, 2], [3, 4]], [[2]], [[1, 0], [0, 2]]):
    print(g, "->", projection_area_one_pass(g))

[[1, 2], [3, 4]] -> 17
[[2]] -> 5
[[1, 0], [0, 2]] -> 8


## 7. TRACE từng bước

### 7.1. Bảng trace cách 3 với `grid = [[1,2],[3,4]]`

| i | j | grid[i][j] | top +? | row_max | grid[j][i] | col_max | total sau hàng |
|---:|---:|---:|:---:|---:|---:|---:|---:|
| 0 | 0 | 1 | +1 | 1 | 1 | 1 | |
| 0 | 1 | 2 | +1 | 2 | 3 | 3 | 2 + 2 + 3 = **7** |
| 1 | 0 | 3 | +1 | 3 | 2 | 2 | |
| 1 | 1 | 4 | +1 | 4 | 4 | 4 | 7 + 2 + 4 + 4 = **17** |

In [10]:
def trace_projection(grid):
    n = len(grid)
    total = 0
    for i in range(n):
        row_max = col_max = 0
        for j in range(n):
            add = 1 if grid[i][j] > 0 else 0
            total += add
            row_max = max(row_max, grid[i][j])
            col_max = max(col_max, grid[j][i])
            print(f"i={i} j={j}: grid[i][j]={grid[i][j]} (top+{add}), row_max={row_max} | grid[j][i]={grid[j][i]}, col_max={col_max}")
        total += row_max + col_max
        print(f"  -> hết hàng {i}: cộng row_max {row_max} + col_max {col_max}, total = {total}")
    print("KẾT QUẢ:", total)
    return total

_ = trace_projection([[1, 2], [3, 4]])

i=0 j=0: grid[i][j]=1 (top+1), row_max=1 | grid[j][i]=1, col_max=1
i=0 j=1: grid[i][j]=2 (top+1), row_max=2 | grid[j][i]=3, col_max=3
  -> hết hàng 0: cộng row_max 2 + col_max 3, total = 7
i=1 j=0: grid[i][j]=3 (top+1), row_max=3 | grid[j][i]=2, col_max=2
i=1 j=1: grid[i][j]=4 (top+1), row_max=4 | grid[j][i]=4, col_max=4
  -> hết hàng 1: cộng row_max 4 + col_max 4, total = 17
KẾT QUẢ: 17


### 7.2. Trace có ô 0

`[[1,0],[0,2]]`: ô `(0,1)` và `(1,0)` không cộng vào top, nhưng max hàng/cột vẫn tính đúng.

In [11]:
_ = trace_projection([[1, 0], [0, 2]])

i=0 j=0: grid[i][j]=1 (top+1), row_max=1 | grid[j][i]=1, col_max=1
i=0 j=1: grid[i][j]=0 (top+0), row_max=1 | grid[j][i]=0, col_max=1
  -> hết hàng 0: cộng row_max 1 + col_max 1, total = 3
i=1 j=0: grid[i][j]=0 (top+0), row_max=0 | grid[j][i]=0, col_max=0
i=1 j=1: grid[i][j]=2 (top+1), row_max=2 | grid[j][i]=2, col_max=2
  -> hết hàng 1: cộng row_max 2 + col_max 2, total = 8
KẾT QUẢ: 8


### 7.3. Đối chiếu từng mặt giữa mô phỏng và công thức

In riêng diện tích từng mặt của cách mô phỏng (set) và cách công thức, để thấy chúng khớp **từng thành phần**, không chỉ khớp tổng.

In [12]:
def faces_simulate(grid):
    xy, yz, zx = set(), set(), set()
    n = len(grid)
    for x in range(n):
        for y in range(n):
            for z in range(grid[x][y]):
                xy.add((x, y)); yz.add((y, z)); zx.add((z, x))
    return len(xy), len(yz), len(zx)

def faces_formula(grid):
    top = sum(v > 0 for row in grid for v in row)
    cols = sum(max(col) for col in zip(*grid))     # yz: mỗi cột y góp max cột
    rows = sum(max(row) for row in grid)           # zx: mỗi hàng x góp max hàng
    return top, cols, rows

for g in ([[1, 2], [3, 4]], [[1, 0], [0, 2]], [[0, 0], [0, 5]]):
    print(g, "| mô phỏng (xy,yz,zx):", faces_simulate(g), "| công thức:", faces_formula(g))

[[1, 2], [3, 4]] | mô phỏng (xy,yz,zx): (4, 7, 6) | công thức: (4, 7, 6)
[[1, 0], [0, 2]] | mô phỏng (xy,yz,zx): (2, 3, 3) | công thức: (2, 3, 3)
[[0, 0], [0, 5]] | mô phỏng (xy,yz,zx): (1, 5, 5) | công thức: (1, 5, 5)


## 8. Code nộp LeetCode

In [13]:
class Solution:
    def projectionArea(self, grid: list[list[int]]) -> int:
        top = sum(v > 0 for row in grid for v in row)
        front = sum(max(row) for row in grid)
        side = sum(max(col) for col in zip(*grid))
        return top + front + side

sol = Solution()
print(sol.projectionArea([[1, 2], [3, 4]]), sol.projectionArea([[2]]), sol.projectionArea([[1, 0], [0, 2]]))

17 5 8


### 8.1. Bản nộp một vòng lặp

In [14]:
class SolutionOnePass:
    def projectionArea(self, grid: list[list[int]]) -> int:
        n = len(grid)
        total = 0
        for i in range(n):
            row_max = col_max = 0
            for j in range(n):
                if grid[i][j]:
                    total += 1
                row_max = max(row_max, grid[i][j])
                col_max = max(col_max, grid[j][i])
            total += row_max + col_max
        return total

print(SolutionOnePass().projectionArea([[1, 2], [3, 4]]))

17


`if grid[i][j]:` — số `0` được coi là **False**, số khác 0 là **True** (truthiness). Viết vậy tương đương `if grid[i][j] > 0` vì giá trị không âm.

## 9. Vì sao công thức đúng?

Tập khối lập phương: `C = {(x, y, z) : 0 ≤ z < grid[x][y]}`.

- **Bóng xy** = `{(x, y) : ∃z, (x,y,z) ∈ C}` = `{(x, y) : grid[x][y] ≥ 1}` → đếm số ô dương.
- **Bóng yz** = `{(y, z) : ∃x, z < grid[x][y]}` = `{(y, z) : z < max_x grid[x][y]}` → với mỗi `y` có đúng `max_x grid[x][y]` giá trị `z` → tổng max **cột**.
- **Bóng zx** = `{(z, x) : ∃y, z < grid[x][y]}` → với mỗi `x` có `max_y grid[x][y]` giá trị → tổng max **hàng**.

Mấu chốt: "tồn tại một tháp cao hơn z" ⇔ "tháp cao nhất cao hơn z". Bóng chiếu = **phép lấy max** theo hướng nhìn.

**Invariant cách 3:** sau khi xử lý xong hàng `i`, `total` = (số ô dương trong hàng 0..i) + Σ max hàng 0..i + Σ max cột 0..i.

## 10. Độ phức tạp và so sánh

`n` = kích thước lưới, `H` = chiều cao tối đa (≤ 50).

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Ghi chú |
|---|---|---|---|---|
| 1. Mô phỏng set | thêm bóng từng khối | `O(n²·H)` | `O(n² + n·H)` | Trực quan, kiểm chứng tốt |
| 2. Công thức 3 lượt | đếm + Σmax hàng + Σmax cột | `O(n²)` | `O(n)` (zip tạo các tuple cột) | Gọn nhất |
| 3. Một vòng đôi | `grid[i][j]` & `grid[j][i]` | `O(n²)` | `O(1)` | Tận dụng lưới vuông |

Không thể dưới `O(n²)` vì phải xem mọi ô ít nhất một lần.

## 11. Bộ test: mẫu đề, biên, đối chiếu

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

methods = [projection_area_simulate, projection_area_formula, projection_area_one_pass,
           Solution().projectionArea, SolutionOnePass().projectionArea]

cases = [
    ("đề VD1", [[1, 2], [3, 4]], 17),
    ("đề VD2", [[2]], 5),
    ("đề VD3", [[1, 0], [0, 2]], 8),
    ("toàn 0", [[0, 0], [0, 0]], 0),
    ("1 ô = 0", [[0]], 0),
    ("1 ô cao 50", [[50]], 1 + 50 + 50),
    ("hàng giống nhau", [[1, 1, 1], [1, 1, 1], [1, 1, 1]], 9 + 3 + 3),
    ("đường chéo", [[3, 0, 0], [0, 3, 0], [0, 0, 3]], 3 + 9 + 9),
]
for name, g, exp in cases:
    for f in methods:
        assert f(g) == exp, (name, f)
    check(name, projection_area_formula(g), exp)

PASS  đề VD1: 17
PASS  đề VD2: 5
PASS  đề VD3: 8
PASS  toàn 0: 0
PASS  1 ô = 0: 0
PASS  1 ô cao 50: 101
PASS  hàng giống nhau: 15
PASS  đường chéo: 21


In [16]:
import random
random.seed(883)
for _ in range(300):
    n = random.randint(1, 6)
    g = [[random.choice([0, 0, 1, 2, 3, 7]) for _ in range(n)] for _ in range(n)]
    expected = projection_area_simulate(g)
    for f in methods:
        assert f(g) == expected
print("300 lưới ngẫu nhiên: 5 cách luôn trùng kết quả mô phỏng.")

big = [[50] * 50 for _ in range(50)]
print("Lưới lớn nhất 50x50 toàn 50:", Solution().projectionArea(big), "=", 2500 + 50 * 50 + 50 * 50)

300 lưới ngẫu nhiên: 5 cách luôn trùng kết quả mô phỏng.
Lưới lớn nhất 50x50 toàn 50: 7500 = 7500


## 12. Lỗi người mới hay gặp

### Lỗi 1: Đếm mọi ô cho mặt trên (quên ô 0)
```python
top = n * n          # SAI khi có ô 0
```
### Lỗi 2: Cộng **tổng** hàng thay vì **max** hàng
Bóng không cộng dồn chiều cao — tháp thấp bị tháp cao che.
### Lỗi 3: Tính max cột bằng `max(grid[j])` (thực ra là max hàng `j`)
Nhầm hàng/cột → đếm max hàng hai lần, mất max cột.

In [17]:
def wrong_all_cells(grid):
    n = len(grid)
    return n * n + sum(max(r) for r in grid) + sum(max(c) for c in zip(*grid))

def wrong_sum_instead_of_max(grid):
    top = sum(v > 0 for r in grid for v in r)
    return top + sum(sum(r) for r in grid) + sum(sum(c) for c in zip(*grid))

def wrong_rows_twice(grid):
    n = len(grid)
    top = sum(v > 0 for r in grid for v in r)
    return top + sum(max(grid[i]) for i in range(n)) + sum(max(grid[j]) for j in range(n))

g = [[1, 0], [0, 2]]
print("Đúng:", Solution().projectionArea(g))
print("Lỗi 1 (đếm cả ô 0):", wrong_all_cells(g))
g = [[1, 2], [3, 4]]
print("Đúng:", Solution().projectionArea(g))
print("Lỗi 2 (tổng thay max):", wrong_sum_instead_of_max(g))
print("Lỗi 3 (max hàng 2 lần):", wrong_rows_twice(g))

Đúng: 8
Lỗi 1 (đếm cả ô 0): 10
Đúng: 17
Lỗi 2 (tổng thay max): 24
Lỗi 3 (max hàng 2 lần): 16


### Lỗi 4: Dùng `zip(grid)` thiếu dấu `*`
`zip(grid)` chỉ bọc mỗi hàng vào một tuple 1 phần tử — **không** chuyển vị.

In [18]:
print("zip(grid)  :", list(zip([[1, 2], [3, 4]])))
print("zip(*grid) :", list(zip(*[[1, 2], [3, 4]])))

zip(grid)  : [([1, 2],), ([3, 4],)]
zip(*grid) : [(1, 3), (2, 4)]


## 13. Mẹo nhận dạng pattern

- Bài "nhìn từ một phía", "bóng", "skyline", "đường chân trời" → **max theo hướng nhìn**.
- Bài 3D trên lưới với `grid[i][j]` = chiều cao → hầu như luôn quy về **phép tính trên từng ô và các ô kề** (không cần mô phỏng từng khối).
- Cần cả hàng và cột của ma trận vuông → `zip(*grid)` hoặc mẹo `grid[j][i]` trong cùng vòng.
- Trước khi nghĩ công thức, viết **mô phỏng nhỏ** để kiểm chứng — rất hữu ích cho bài hình học.

Bài liên quan: 892 Surface Area of 3D Shapes, 807 Max Increase to Keep City Skyline, 463 Island Perimeter.

## 14. Biến thể

### 14.1. Chỉ một mặt: "skyline nhìn từ phía trước" (list max mỗi cột)
### 14.2. Lưới chữ nhật `m × n` (không vuông)
Cách 2 với `zip(*grid)` vẫn chạy; cách 3 (mẹo `grid[j][i]`) thì **không** dùng được nữa.

In [19]:
def skyline_front(grid):
    return [max(col) for col in zip(*grid)]

def projection_area_rect(grid):
    top = sum(v > 0 for r in grid for v in r)
    return top + sum(max(r) for r in grid) + sum(max(c) for c in zip(*grid))

print("Skyline nhìn theo cột của [[3,0,8,4],[2,4,5,7],[9,2,6,3],[0,3,1,0]]:",
      skyline_front([[3, 0, 8, 4], [2, 4, 5, 7], [9, 2, 6, 3], [0, 3, 1, 0]]))
rect = [[1, 2, 3], [0, 0, 4]]     # 2 x 3
print("Lưới 2x3:", projection_area_rect(rect), "| mô phỏng:", end=" ")
xy, yz, zx = set(), set(), set()
for x in range(2):
    for y in range(3):
        for z in range(rect[x][y]):
            xy.add((x, y)); yz.add((y, z)); zx.add((z, x))
print(len(xy) + len(yz) + len(zx))

Skyline nhìn theo cột của [[3,0,8,4],[2,4,5,7],[9,2,6,3],[0,3,1,0]]: [9, 4, 8, 7]
Lưới 2x3: 18 | mô phỏng: 18


### 14.3. Bài 807 (ý tưởng): tăng chiều cao mỗi tháp tối đa mà **không đổi** skyline hai hướng
Mỗi ô có thể nâng lên `min(max hàng i, max cột j)`. Lại là max theo hàng/cột!

In [20]:
def max_increase_keeping_skyline(grid):
    row_max = [max(r) for r in grid]
    col_max = [max(c) for c in zip(*grid)]
    return sum(min(row_max[i], col_max[j]) - grid[i][j]
               for i in range(len(grid)) for j in range(len(grid[0])))

print(max_increase_keeping_skyline([[3, 0, 8, 4], [2, 4, 5, 7], [9, 2, 6, 3], [0, 3, 1, 0]]))   # 35

35


## 15. Bài tập tự luyện

Hàm TODO chỉ định nghĩa, không được gọi khi chạy notebook.

### Bài 1 — Tính tay
Tính diện tích chiếu của `[[2,2,2],[2,1,2],[2,2,2]]` theo 3 mặt riêng rồi cộng.

### Bài 2 — Dự đoán output
`list(zip(*[[1,2,3],[4,5,6]]))` in ra gì? `sum(x > 2 for x in [1,5,2,7])` bằng bao nhiêu?

### Bài 3 — Đếm ô dương

In [21]:
def count_positive(grid) -> int:
    # TODO: đếm số ô > 0
    pass

# assert count_positive([[1,0],[0,2]]) == 2
# assert count_positive([[0]]) == 0

### Bài 4 — Tự viết lại `projectionArea` không dùng `zip`

In [22]:
def projection_area_exercise(grid) -> int:
    # TODO: chỉ dùng range và chỉ số
    pass

# assert projection_area_exercise([[1,2],[3,4]]) == 17
# assert projection_area_exercise([[1,0],[0,2]]) == 8

### Bài 5 — Biến thể: diện tích chiếu khi nhìn **chỉ từ trên và từ một phía cột** (bỏ phía hàng), lưới chữ nhật

In [23]:
def top_and_side(grid) -> int:
    # TODO
    pass

# assert top_and_side([[1,2,3],[0,0,4]]) == 4 + (1 + 2 + 4)

### Bài 6 — Mini challenge: các tháp "lộ mặt" khi nhìn từ phía cột
Cho lưới, trả về số ô của mặt **xy** mà vẫn **nhìn thấy được từ phía trước** — tức tháp ở ô `(i, j)` cao bằng max cột `j` (không bị tháp nào trong cùng cột che hoàn toàn). Ô 0 không tính.

In [24]:
def visible_from_front(grid) -> int:
    # TODO: so sánh grid[i][j] với max cột j
    pass

# assert visible_from_front([[1,2],[3,4]]) == 2      # ô (1,0)=3, (1,1)=4
# assert visible_from_front([[2,2],[2,2]]) == 4

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | 3 phép riêng | top = 9 ô dương | max hàng 2,2,2; max cột 2,2,2 → 9+6+6 |
| 2 | `*` bung hàng | zip ghép cùng vị trí | `[(1,4),(2,5),(3,6)]`, 2 |
| 3 | True = 1 | generator lồng | `sum(v > 0 for r in grid for v in r)` |
| 4 | vòng i, j | max cột: `max(grid[i][j] for i in range(n))` | cộng 3 phần |
| 5 | lưới chữ nhật | zip(*grid) cho cột | top + Σ max cột |
| 6 | cần max cột trước | `col_max = [max(c) for c in zip(*grid)]` | đếm `v > 0 and v == col_max[j]` |

## 17. Đáp án tham khảo

**Bài 1:** mọi ô > 0 → top = 9; max mỗi hàng = 2 → 6; max mỗi cột = 2 → 6. Tổng **21**.

**Bài 2:** `[(1, 4), (2, 5), (3, 6)]` và `2`.

In [25]:
def ans_count_positive(grid):
    return sum(v > 0 for r in grid for v in r)

def ans_projection_area(grid):
    n = len(grid)
    top = 0
    for i in range(n):
        for j in range(n):
            if grid[i][j] > 0:
                top += 1
    rows = sum(max(grid[i]) for i in range(n))
    cols = sum(max(grid[i][j] for i in range(n)) for j in range(n))
    return top + rows + cols

def ans_top_and_side(grid):
    return ans_count_positive(grid) + sum(max(c) for c in zip(*grid))

def ans_visible_from_front(grid):
    col_max = [max(c) for c in zip(*grid)]
    return sum(1 for row in grid for j, v in enumerate(row) if v > 0 and v == col_max[j])

assert Solution().projectionArea([[2,2,2],[2,1,2],[2,2,2]]) == 21
assert list(zip(*[[1,2,3],[4,5,6]])) == [(1,4),(2,5),(3,6)]
assert sum(x > 2 for x in [1,5,2,7]) == 2
assert ans_count_positive([[1,0],[0,2]]) == 2
assert ans_count_positive([[0]]) == 0
assert ans_projection_area([[1,2],[3,4]]) == 17
assert ans_projection_area([[1,0],[0,2]]) == 8
assert ans_top_and_side([[1,2,3],[0,0,4]]) == 4 + (1 + 2 + 4)
assert ans_visible_from_front([[1,2],[3,4]]) == 2
assert ans_visible_from_front([[2,2],[2,2]]) == 4
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


`enumerate(row)` trả về từng cặp `(chỉ số, giá trị)` — tiện khi vừa cần giá trị vừa cần biết đang ở cột nào.

## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def projectionArea(self, grid):
        top   = sum(v > 0 for row in grid for v in row)   # nhìn từ trên: ô có tháp
        front = sum(max(row) for row in grid)             # mỗi hàng: tháp cao nhất
        side  = sum(max(col) for col in zip(*grid))       # mỗi cột: tháp cao nhất
        return top + front + side
```

**Câu nhớ:** *"Bóng = max theo hướng nhìn; nhìn từ trên thì chỉ cần có hay không."*

**Checklist:**
- [ ] Mặt trên đếm **ô > 0**, không phải `n*n`?
- [ ] Hai mặt bên dùng **max**, không dùng sum?
- [ ] Một mặt theo **hàng**, một mặt theo **cột** (không lặp hàng hai lần)?
- [ ] `zip(*grid)` có dấu `*`?
- [ ] Test `[[0]]`, lưới toàn 0, ví dụ đề có ô 0?